In [ ]:
import copy
import os
import pickle
import numpy as np
from ase.io import read, write

def extract_db_info(atoms, info_keys=[]):
    num_atoms = len(atoms)
    charges = atoms.get_atomic_numbers().tolist()
    positions = atoms.get_positions().tolist()
    rxn = f"rxn{atoms.info['rxn']:04d}"
    additional_info = {key: atoms.info[key] for key in info_keys if key in atoms.info}
    return {
        "num_atoms": num_atoms,
        "charges": charges,
        "positions": positions,
        "rxn": rxn,
        **additional_info
    }

def create_database(data_path, save_path, info_keys):
    db = read(data_path, ":")
    r, ts, p = db[0::3], db[1::3], db[2::3]

    print(f"Number of reactions: {len(r)}")

    database = {
        "reactant": {},
        "transition_state": {},
        "product": {},
        "single_fragment": [1 for _ in range(len(r))],
    }

    for rr, tsts, pp in zip(r, ts, p):
        for key, atoms in zip(["reactant", "transition_state", "product"], [rr, tsts, pp]):
            info = extract_db_info(atoms, info_keys)
            for k, v in info.items():
                if k not in database[key]:
                    database[key][k] = []
                database[key][k].append(v)

    os.makedirs(save_path, exist_ok=True)
    with open(f"{save_path}/all.pkl", "wb") as f:
        pickle.dump(database, f)
        
    return database

def create_split_dataset(data, split_rxn, save_path):
    print("Creating split dataset...")

    rnxs = [int(rxn[3:]) for rxn in data["reactant"]["rxn"]]

    train_idx_given_rxn = set([rxn for i, rxn in enumerate(rnxs) if rxn in split_rxn])
    print(f"{len(train_idx_given_rxn)} reactions in the data set given reactions index")

    train_idx = [i for i, rxn in enumerate(rnxs) if rxn in split_rxn]
    test_idx = [i for i in range(len(rnxs)) if i not in train_idx]

    data_train = copy.deepcopy(data)
    data_val = copy.deepcopy(data)

    data_train["use_ind"] = train_idx
    data_val["use_ind"] = test_idx

    print(
        "Train set size:",
        len(train_idx),
        f"({len(train_idx) / (len(rnxs)) * 100:.2f}%)",
    )
    print(
        "Validation set size:",
        len(test_idx),
        f"({len(test_idx) / (len(rnxs)) * 100:.2f}%)\n",
    )

    os.makedirs(save_path, exist_ok=True)
    with open(f"{save_path}/train_rpsb_all.pkl", "wb") as f:
        pickle.dump(data_train, f)

    with open(f"{save_path}/valid_rpsb_all.pkl", "wb") as f:
        pickle.dump(data_val, f)

In [ ]:
# keys in database: keys = ["reactant", "transition_state", "product", "single_fragment"]
# single fragment mask: 0: multi fragment 1: single fragment
# keys per fragment:
# num_atoms: number of atoms per molecule
# charges: list with atomic charges
# positions: list with atomic positions
# rxn: reaction ID

split = np.load("reactot/data/oa_reactdiff_split.npz")
split_rxn = split["train_idx"]

# Swapped reaction info to include in the database
info_keys = ["swapped_type", "swapped_index", "swap"]
# TM reaction info to include in the database
info_keys = ["tm"]

save_path = "../data/xxx"

for sub_folder in ["xxx"]:
    data_path = f"gathered_jobs/{sub_folder}/irc_db_filtered_aligned.xyz"
    database = create_database(data_path, f"{save_path}/{sub_folder}", info_keys)
    create_split_dataset(database, split_rxn=split_rxn, save_path=f"{save_path}/{sub_folder}")